
# 014 Top-Decoding Archetype Clustering Validation — Spatial Enabled

This revised notebook enables the clustering validation for **spatial AA** by clustering **condition-subject observations** rather than nodes.

The earlier notebook skipped spatial AA because it treated nodes as samples, and nodes do not have intact/word/rest labels. Here, each condition-subject observation becomes one feature vector built from selected archetype expression timecourses.

For each analysis type, \(K\), condition, and top-\(m\), the notebook compares:

\[
\text{top-decoding archetype subset} \quad \text{vs.} \quad \text{random archetype subsets from the same model and }K.
\]

Spatial AA uses `sXC[:, selected_archetypes]`; temporal AA uses `S[selected_archetypes, :].T`.


In [ ]:

# ============================================================
# SETTINGS
# ============================================================

FIT_SCOPE = "across"
MSAA_RESULTS_DIR = "."
DECODING_DIR_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}"

ANALYSIS_TYPES = ["spatial", "temporal"]
CONDITIONS = ["intact", "word", "rest"]

K_VALUES_BY_ANALYSIS = {
    "spatial": [5, 7, 10, 14, 21, 25, 35, 40, 42, 44, 45, 46, 48, 50, 52, 53, 54, 55, 56, 58, 60, 63, 65, 70, 75, 88, 100, 105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500, 600, 700],
    "temporal": [2, 3, 5, 6, 8, 9, 10, 11, 14, 15, 17, 18, 20, 23, 25, 27, 30, 35, 38, 40, 45, 50, 54, 55, 60, 65, 75, 81, 90, 100, 108, 200, 300],
}

TOP_M_VALUES = [1, 3, 5, 7, 10, 15, 20, 25, 30, 40, 50]

N_RANDOM_SUBSETS = 500
RANDOM_SEED = 0

N_CLUSTERS = 3
CLUSTER_METHOD = "spectral"  # "spectral" or "kmeans"
N_INIT = 50

FEATURE_MODE = "flatten"  # "flatten" or "summary"
SCALE_FEATURES = True
USE_PCA = False
PCA_N_COMPONENTS = 20

SKIP_IF_TOP_M_GE_N_ARCHETYPES = True

SPATIAL_DENOMINATOR = 700
TEMPORAL_DENOMINATOR = 300

FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "014_spatial_enabled_top_decoding_clustering_validation"
SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300
FIGSIZE = (8.5, 5.2)

COND_COLORS = {"intact": "purple", "word": "green", "rest": "black"}


In [ ]:

# ============================================================
# IMPORTS AND HELPERS
# ============================================================

%matplotlib inline

from pathlib import Path
import re, ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

_fig_counter = 0

def _safe_name(name):
    name = str(name).replace(" ", "_").replace("/", "-").replace("|", "_")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:180] if name else "figure"

def save_current_fig(name):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    _fig_counter += 1
    out = FIG_DIR / f"{_fig_counter:03d}_{_safe_name(name)}.{FIG_FORMAT}"
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def to_float_array(x):
    return np.asarray(x, dtype=float)

def component_ratio(K, analysis_type):
    return float(K) / (SPATIAL_DENOMINATOR if analysis_type == "spatial" else TEMPORAL_DENOMINATOR)

print("Figure directory:", FIG_DIR)


In [ ]:

# ============================================================
# LOAD MSAA FITS AND TOP-M SUMMARIES
# ============================================================

def find_msaa_npz(analysis_type, fit_scope, K, base_dir=MSAA_RESULTS_DIR):
    base_dir = Path(base_dir)
    candidates = []
    for p in base_dir.rglob("*.npz"):
        name = p.name.lower()
        if analysis_type.lower() in name and fit_scope.lower() in name and f"k{K}" in name:
            candidates.append(p)
    if len(candidates) == 0:
        for p in base_dir.rglob("*.npz"):
            name = p.name.lower()
            if analysis_type.lower() in name and f"k{K}" in name:
                candidates.append(p)
    if len(candidates) == 0:
        print(f"No npz found for {analysis_type} {fit_scope} K={K}")
        return None
    candidates = sorted(candidates, key=lambda p: (len(str(p)), str(p)))
    if len(candidates) > 1:
        print(f"Multiple candidates for {analysis_type} {fit_scope} K={K}; using:", candidates[0])
    return candidates[0]

def load_npz_as_dict(path):
    z = np.load(path, allow_pickle=True)
    out = {}
    for key in z.files:
        val = z[key]
        if hasattr(val, "shape") and val.shape == () and val.dtype == object:
            val = val.item()
        out[key] = val
    return out

def unpack_results_subj(d):
    for key in ["results_subj", "results", "subject_results", "subj_results"]:
        if key in d:
            obj = d[key]
            if isinstance(obj, list):
                return obj
            if isinstance(obj, np.ndarray) and obj.dtype == object:
                return list(obj)
    if "sXC" in d and "S" in d:
        return [{"sXC": d["sXC"], "S": d["S"]}]
    for val in d.values():
        if isinstance(val, np.ndarray) and val.dtype == object:
            maybe = list(val)
            if len(maybe) and isinstance(maybe[0], dict):
                return maybe
    raise ValueError("Could not unpack subject results from npz.")

def infer_condition_labels(d, results_subj):
    for key in ["condition_labels_str", "condition_labels", "labels", "conds", "conditions"]:
        if key in d:
            labs = np.asarray(d[key]).astype(str)
            if len(labs) == len(results_subj):
                return labs
    n = len(results_subj)
    if n % len(CONDITIONS) == 0:
        block = n // len(CONDITIONS)
        print(f"Inferred condition labels as equal blocks of {block}: {CONDITIONS}")
        return np.asarray(sum(([c] * block for c in CONDITIONS), []))
    raise ValueError("Could not infer condition labels. Add condition_labels_str to the npz or edit infer_condition_labels().")

def load_msaa_results(analysis_type, fit_scope, K):
    path = find_msaa_npz(analysis_type, fit_scope, K)
    if path is None:
        return None, None, None
    d = load_npz_as_dict(path)
    results_subj = unpack_results_subj(d)
    condition_labels = infer_condition_labels(d, results_subj)
    return results_subj, condition_labels, path

def parse_selected_archetypes(x):
    if isinstance(x, list):
        return [int(v) for v in x]
    if isinstance(x, np.ndarray):
        return [int(v) for v in x.tolist()]
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            return [int(v) for v in ast.literal_eval(x)]
        except Exception:
            return [int(v) for v in re.findall(r"-?\d+", x)]
    return []

def standardize_topm_df(df, analysis_type, fit_scope):
    df = df.copy()
    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "err" not in df.columns:
        rename["sem_accuracy"] = "err"
    if "sem" in df.columns and "err" not in df.columns:
        rename["sem"] = "err"
    df = df.rename(columns=rename)
    if "analysis_type" not in df.columns:
        df["analysis_type"] = analysis_type
    if "fit_scope" not in df.columns:
        df["fit_scope"] = fit_scope
    df["analysis_type"] = df["analysis_type"].astype(str)
    df["fit_scope"] = df["fit_scope"].astype(str)
    df["condition"] = df["condition"].astype(str)
    df["K"] = df["K"].astype(int)
    df["top_m"] = df["top_m"].astype(int)
    if "selected_archetypes" not in df.columns:
        raise ValueError("topm_summary.csv must contain selected_archetypes column.")
    df["selected_archetype_list"] = df["selected_archetypes"].apply(parse_selected_archetypes)
    df["component_ratio"] = df.apply(lambda r: component_ratio(r["K"], r["analysis_type"]), axis=1)
    return df[(df["analysis_type"] == analysis_type) & (df["fit_scope"] == fit_scope)].copy()

def load_topm_summary(analysis_type, fit_scope):
    d = Path(DECODING_DIR_TEMPLATE.format(analysis_type=analysis_type, fit_scope=fit_scope))
    path = d / "topm_summary.csv"
    if not path.exists():
        print("Missing:", path)
        return pd.DataFrame()
    print("Loaded:", path)
    return standardize_topm_df(pd.read_csv(path), analysis_type, fit_scope)

topm_dfs = []
for analysis_type in ANALYSIS_TYPES:
    df = load_topm_summary(analysis_type, FIT_SCOPE)
    if len(df):
        topm_dfs.append(df)

topm_df = pd.concat(topm_dfs, ignore_index=True) if topm_dfs else pd.DataFrame()
topm_df = topm_df[topm_df["top_m"].isin(TOP_M_VALUES)].copy()

print("topm rows:", len(topm_df))
display(topm_df.head())


In [ ]:

# ============================================================
# CONDITION-SUBJECT FEATURE MATRIX
# ============================================================

def n_archetypes_in_fit(results_subj, analysis_type):
    first = results_subj[0]
    if analysis_type == "spatial":
        return to_float_array(first["sXC"]).shape[1]
    if analysis_type == "temporal":
        return to_float_array(first["S"]).shape[0]
    raise ValueError("analysis_type must be spatial or temporal")

def get_expression_matrix_for_observation(sub_result, analysis_type, selected_archetypes):
    selected_archetypes = [int(a) for a in selected_archetypes]
    if analysis_type == "spatial":
        sXC = to_float_array(sub_result["sXC"])
        return sXC[:, selected_archetypes]
    if analysis_type == "temporal":
        S = to_float_array(sub_result["S"])
        return S[selected_archetypes, :].T
    raise ValueError("analysis_type must be spatial or temporal")

def summarize_expression_matrix(M):
    M = np.asarray(M, dtype=float)
    return np.concatenate([M.mean(axis=0), M.std(axis=0), M.min(axis=0), M.max(axis=0)])

def build_observation_feature_matrix(results_subj, analysis_type, selected_archetypes):
    rows = []
    for sub in results_subj:
        M = get_expression_matrix_for_observation(sub, analysis_type, selected_archetypes)
        M = np.nan_to_num(M, nan=0.0, posinf=0.0, neginf=0.0)
        if FEATURE_MODE == "flatten":
            rows.append(M.ravel())
        elif FEATURE_MODE == "summary":
            rows.append(summarize_expression_matrix(M))
        else:
            raise ValueError("FEATURE_MODE must be 'flatten' or 'summary'.")
    X = np.vstack(rows)
    if SCALE_FEATURES:
        X = StandardScaler().fit_transform(X)
    if USE_PCA and X.shape[1] > PCA_N_COMPONENTS:
        n_comp = min(PCA_N_COMPONENTS, X.shape[0] - 1, X.shape[1])
        X = PCA(n_components=n_comp, random_state=RANDOM_SEED).fit_transform(X)
    return X


In [ ]:

# ============================================================
# CLUSTERING METRICS
# ============================================================

def cluster_samples(X, n_clusters=N_CLUSTERS, seed=RANDOM_SEED):
    X = np.asarray(X, dtype=float)
    if CLUSTER_METHOD == "kmeans":
        return KMeans(n_clusters=n_clusters, random_state=seed, n_init=N_INIT).fit_predict(X)
    if CLUSTER_METHOD == "spectral":
        sim = np.corrcoef(X)
        sim = np.nan_to_num(sim, nan=0.0, posinf=0.0, neginf=0.0)
        sim = np.clip(sim, -1.0, 1.0)
        aff = (sim + 1.0) / 2.0
        np.fill_diagonal(aff, 1.0)
        return SpectralClustering(
            n_clusters=n_clusters,
            affinity="precomputed",
            assign_labels="kmeans",
            random_state=seed,
        ).fit_predict(aff)
    raise ValueError("CLUSTER_METHOD must be 'kmeans' or 'spectral'.")

def cluster_purity(true_labels, pred_labels):
    true_labels = np.asarray(true_labels)
    pred_labels = np.asarray(pred_labels)
    true_vals = pd.unique(true_labels)
    pred_vals = pd.unique(pred_labels)
    table = np.zeros((len(pred_vals), len(true_vals)), dtype=int)
    for i, p in enumerate(pred_vals):
        for j, t in enumerate(true_vals):
            table[i, j] = np.sum((pred_labels == p) & (true_labels == t))
    row_ind, col_ind = linear_sum_assignment(-table)
    return float(table[row_ind, col_ind].sum() / len(true_labels))

def cluster_balance(pred_labels):
    _, counts = np.unique(pred_labels, return_counts=True)
    if len(counts) <= 1:
        return 0.0
    expected = len(pred_labels) / len(counts)
    imbalance = np.sum(np.abs(counts - expected)) / (2 * len(pred_labels) * (1 - 1 / len(counts)))
    return float(1 - imbalance)

def nearest_centroid_accuracy(X, labels):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    unique = pd.unique(labels)
    correct = 0
    for i in range(len(X)):
        train = np.ones(len(X), dtype=bool)
        train[i] = False
        cents = []
        labs = []
        for lab in unique:
            mask = (labels == lab) & train
            if np.sum(mask):
                cents.append(X[mask].mean(axis=0))
                labs.append(lab)
        cents = np.vstack(cents)
        pred = labs[int(np.argmin(cdist(X[[i]], cents).ravel()))]
        correct += int(pred == labels[i])
    return float(correct / len(labels))

def evaluate_clustering(X, labels, n_clusters=N_CLUSTERS, seed=RANDOM_SEED):
    pred = cluster_samples(X, n_clusters=n_clusters, seed=seed)
    purity = cluster_purity(labels, pred)
    balance = cluster_balance(pred)
    out = {
        "purity": purity,
        "balance": balance,
        "purity_x_balance": float(purity * balance),
        "ari": float(adjusted_rand_score(labels, pred)),
        "nmi": float(normalized_mutual_info_score(labels, pred)),
        "nearest_centroid_accuracy": nearest_centroid_accuracy(X, labels),
    }
    try:
        out["silhouette_true_labels"] = float(silhouette_score(X, labels))
    except Exception:
        out["silhouette_true_labels"] = np.nan
    try:
        out["silhouette_cluster_labels"] = float(silhouette_score(X, pred))
    except Exception:
        out["silhouette_cluster_labels"] = np.nan
    return out, pred


In [ ]:

# ============================================================
# RUN TOP-DECODING VS RANDOM-SUBSET CLUSTERING VALIDATION
# ============================================================

rows = []
skip_rows = []
example_store = {}

for analysis_type in ANALYSIS_TYPES:
    for K in K_VALUES_BY_ANALYSIS.get(analysis_type, []):
        results_subj, condition_labels, npz_path = load_msaa_results(analysis_type, FIT_SCOPE, K)
        if results_subj is None:
            skip_rows.append({"analysis_type": analysis_type, "K": K, "reason": "missing_fit"})
            continue

        n_arch = n_archetypes_in_fit(results_subj, analysis_type)
        labels = np.asarray(condition_labels).astype(str)

        for condition in CONDITIONS:
            for top_m in TOP_M_VALUES:
                selected_rows = topm_df[
                    (topm_df["analysis_type"] == analysis_type) &
                    (topm_df["fit_scope"] == FIT_SCOPE) &
                    (topm_df["K"] == int(K)) &
                    (topm_df["condition"] == condition) &
                    (topm_df["top_m"] == int(top_m))
                ]
                if len(selected_rows) == 0:
                    skip_rows.append({"analysis_type": analysis_type, "K": K, "condition": condition, "top_m": top_m, "reason": "no_topm_row"})
                    continue

                row = selected_rows.iloc[0]
                selected = [int(a) for a in row["selected_archetype_list"] if int(a) < n_arch]
                actual_m = len(selected)

                if actual_m == 0:
                    skip_rows.append({"analysis_type": analysis_type, "K": K, "condition": condition, "top_m": top_m, "reason": "empty_selected"})
                    continue

                if SKIP_IF_TOP_M_GE_N_ARCHETYPES and actual_m >= n_arch:
                    skip_rows.append({
                        "analysis_type": analysis_type, "K": K, "condition": condition, "top_m": top_m,
                        "actual_m": actual_m, "n_archetypes": n_arch, "reason": "top_m_ge_n_archetypes"
                    })
                    continue

                X_top = build_observation_feature_matrix(results_subj, analysis_type, selected)

                base = {
                    "analysis_type": analysis_type,
                    "fit_scope": FIT_SCOPE,
                    "K": int(K),
                    "component_ratio": component_ratio(K, analysis_type),
                    "condition": condition,
                    "top_m": int(top_m),
                    "actual_m": int(actual_m),
                    "n_archetypes": int(n_arch),
                    "n_observations": int(len(labels)),
                    "n_clusters": int(N_CLUSTERS),
                    "decode_mean": float(row["mean"]) if "mean" in row.index else np.nan,
                    "decode_err": float(row["err"]) if "err" in row.index else np.nan,
                    "selected_archetypes": str(selected),
                    "feature_mode": FEATURE_MODE,
                    "npz_path": str(npz_path),
                }

                obs_metrics, obs_pred = evaluate_clustering(X_top, labels, n_clusters=N_CLUSTERS, seed=RANDOM_SEED)
                rows.append({**base, "subset_type": "top_decoding", "iter": -1, **obs_metrics})

                example_store[(analysis_type, int(K), condition, int(top_m))] = {
                    "X": X_top, "labels": labels, "pred": obs_pred,
                    "selected": selected, "metrics": obs_metrics
                }

                rng = np.random.default_rng(RANDOM_SEED + 1000 * int(K) + 10 * int(top_m))
                for ii in range(N_RANDOM_SUBSETS):
                    rand_sel = rng.choice(np.arange(n_arch), size=actual_m, replace=False).tolist()
                    X_rand = build_observation_feature_matrix(results_subj, analysis_type, rand_sel)
                    rand_metrics, _ = evaluate_clustering(X_rand, labels, n_clusters=N_CLUSTERS, seed=RANDOM_SEED + ii)
                    rows.append({**base, "subset_type": "random", "iter": ii, "selected_archetypes": str(rand_sel), **rand_metrics})

clustering_validation_long_df = pd.DataFrame(rows)
clustering_validation_skipped_df = pd.DataFrame(skip_rows)

print("Validation rows:", len(clustering_validation_long_df))
print("Skipped rows:", len(clustering_validation_skipped_df))
display(clustering_validation_long_df.head())
display(clustering_validation_skipped_df.head(30))


In [ ]:

# ============================================================
# SUMMARIZE TOP-DECODING VS RANDOM
# ============================================================

METRICS = ["purity", "balance", "purity_x_balance", "ari", "nmi", "nearest_centroid_accuracy", "silhouette_true_labels", "silhouette_cluster_labels"]

group_cols = [
    "analysis_type", "fit_scope", "K", "component_ratio", "condition", "top_m",
    "actual_m", "n_clusters", "decode_mean", "decode_err", "feature_mode"
]

summary_rows = []

for keys, sub in clustering_validation_long_df.groupby(group_cols):
    key_dict = dict(zip(group_cols, keys))
    obs = sub[sub["subset_type"] == "top_decoding"]
    rnd = sub[sub["subset_type"] == "random"]
    if len(obs) == 0 or len(rnd) == 0:
        continue

    obs_row = obs.iloc[0]
    out = dict(key_dict)

    for metric in METRICS:
        obs_val = float(obs_row[metric])
        rand_vals = rnd[metric].to_numpy(dtype=float)
        rand_vals = rand_vals[np.isfinite(rand_vals)]
        out[f"{metric}_observed"] = obs_val

        if len(rand_vals) == 0 or not np.isfinite(obs_val):
            out[f"{metric}_random_mean"] = np.nan
            out[f"{metric}_random_std"] = np.nan
            out[f"{metric}_z"] = np.nan
            out[f"{metric}_p_high"] = np.nan
            out[f"{metric}_p_low"] = np.nan
            out[f"{metric}_observed_minus_random"] = np.nan
            continue

        out[f"{metric}_random_mean"] = float(np.mean(rand_vals))
        out[f"{metric}_random_std"] = float(np.std(rand_vals))
        out[f"{metric}_z"] = float((obs_val - np.mean(rand_vals)) / (np.std(rand_vals) + 1e-12))
        out[f"{metric}_p_high"] = float((np.sum(rand_vals >= obs_val) + 1) / (len(rand_vals) + 1))
        out[f"{metric}_p_low"] = float((np.sum(rand_vals <= obs_val) + 1) / (len(rand_vals) + 1))
        out[f"{metric}_observed_minus_random"] = float(obs_val - np.mean(rand_vals))

    summary_rows.append(out)

clustering_validation_summary_df = pd.DataFrame(summary_rows)

print("Summary rows:", len(clustering_validation_summary_df))
display(clustering_validation_summary_df.head())
display(clustering_validation_summary_df.sort_values("nmi_observed_minus_random", ascending=False).head(30))


In [ ]:

# ============================================================
# PLOTS: OBSERVED VS RANDOM
# ============================================================

def plot_metric_observed_vs_random(summary_df, metric="nmi", condition="intact", top_m=3):
    sub = summary_df[(summary_df["condition"] == condition) & (summary_df["top_m"] == top_m)].copy()
    if len(sub) == 0:
        print("No rows:", metric, condition, top_m)
        return

    fig, ax = plt.subplots(figsize=FIGSIZE)

    for analysis_type, linestyle, marker in [("spatial", "-", "o"), ("temporal", "--", "s")]:
        a = sub[sub["analysis_type"] == analysis_type].sort_values("component_ratio")
        if len(a) == 0:
            continue

        ax.plot(a["component_ratio"], a[f"{metric}_observed"], linestyle=linestyle, marker=marker, linewidth=2.2, label=f"{analysis_type} top-decoding")
        ax.plot(a["component_ratio"], a[f"{metric}_random_mean"], linestyle=linestyle, linewidth=1.5, alpha=0.45, label=f"{analysis_type} random")
        ax.fill_between(
            a["component_ratio"],
            a[f"{metric}_random_mean"] - 1.96 * a[f"{metric}_random_std"],
            a[f"{metric}_random_mean"] + 1.96 * a[f"{metric}_random_std"],
            alpha=0.12,
        )

        for _, r in a.iterrows():
            z = r.get(f"{metric}_z", np.nan)
            if np.isfinite(z) and abs(z) >= 2:
                ax.text(r["component_ratio"], r[f"{metric}_observed"], f"K={int(r['K'])}", fontsize=8, ha="center", va="bottom")

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel(metric)
    ax.set_title(f"{condition}: {metric}, top-{top_m}\ntop-decoding subsets vs random subsets")
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    save_current_fig(f"clustering_validation_{metric}_{condition}_top{top_m}")
    plt.show()
    plt.close()

for condition in CONDITIONS:
    for top_m in [1, 3, 5, 10]:
        for metric in ["purity", "nmi", "ari", "nearest_centroid_accuracy", "silhouette_true_labels"]:
            plot_metric_observed_vs_random(clustering_validation_summary_df, metric=metric, condition=condition, top_m=top_m)


In [ ]:

# ============================================================
# PLOTS: TOP-DECODING MINUS RANDOM ADVANTAGE
# ============================================================

def plot_metric_advantage(summary_df, metric="nmi", condition="intact"):
    sub = summary_df[summary_df["condition"] == condition].copy()
    if len(sub) == 0:
        print("No rows:", metric, condition)
        return

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    for analysis_type, linestyle, marker in [("spatial", "-", "o"), ("temporal", "--", "s")]:
        for top_m in sorted(sub["top_m"].unique()):
            a = sub[(sub["analysis_type"] == analysis_type) & (sub["top_m"] == top_m)].sort_values("component_ratio")
            if len(a) == 0:
                continue
            ax.plot(
                a["component_ratio"],
                a[f"{metric}_observed_minus_random"],
                linestyle=linestyle,
                marker=marker,
                linewidth=1.8,
                alpha=0.55 + 0.04 * min(top_m, 10),
                label=f"{analysis_type} top-{top_m}",
            )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel(f"Top-decoding minus random {metric}")
    ax.set_title(f"{condition}: clustering advantage over random subsets")
    ax.legend(frameon=False, fontsize=7, ncol=2)
    plt.tight_layout()
    save_current_fig(f"clustering_advantage_{metric}_{condition}")
    plt.show()
    plt.close()

for condition in CONDITIONS:
    for metric in ["purity", "nmi", "ari", "nearest_centroid_accuracy", "silhouette_true_labels"]:
        plot_metric_advantage(clustering_validation_summary_df, metric=metric, condition=condition)


In [ ]:

# ============================================================
# BEST-CASE SIMILARITY MATRICES
# ============================================================

def order_by_labels_and_similarity(sim, labels):
    labels = np.asarray(labels)
    order = []
    for lab in pd.unique(labels):
        idx = np.where(labels == lab)[0]
        if len(idx) == 1:
            order.extend(idx.tolist())
        else:
            sub = sim[np.ix_(idx, idx)]
            local = idx[np.argsort(-sub.mean(axis=1))]
            order.extend(local.tolist())
    order = np.asarray(order)
    labels_sorted = labels[order]
    boundaries = np.where(labels_sorted[1:] != labels_sorted[:-1])[0] + 1
    return order, labels_sorted, boundaries

def plot_similarity_example(key):
    if key not in example_store:
        print("Missing example:", key)
        return
    ex = example_store[key]
    X = ex["X"]
    labels = ex["labels"]
    metrics = ex["metrics"]

    sim = np.corrcoef(X)
    sim = np.nan_to_num(sim, nan=0.0, posinf=0.0, neginf=0.0)
    sim = np.clip(sim, -1, 1)

    order, labels_sorted, boundaries = order_by_labels_and_similarity(sim, labels)

    fig, ax = plt.subplots(figsize=(6.5, 6))
    im = ax.imshow(sim[np.ix_(order, order)], cmap="coolwarm", vmin=-1, vmax=1)

    for b in boundaries:
        ax.axhline(b - 0.5, color="black", linewidth=2)
        ax.axvline(b - 0.5, color="black", linewidth=2)

    ax.set_xticks(np.arange(len(order)))
    ax.set_yticks(np.arange(len(order)))
    ax.set_xticklabels(labels_sorted, rotation=90, fontsize=6)
    ax.set_yticklabels(labels_sorted, fontsize=6)

    for tick, lab in zip(ax.get_xticklabels(), labels_sorted):
        tick.set_color(COND_COLORS.get(str(lab), "gray"))
    for tick, lab in zip(ax.get_yticklabels(), labels_sorted):
        tick.set_color(COND_COLORS.get(str(lab), "gray"))

    ax.set_title(
        f"{key[0]} K={key[1]} | {key[2]} top-{key[3]}\n"
        f"NMI={metrics['nmi']:.2f}, purity={metrics['purity']:.2f}, "
        f"ARI={metrics['ari']:.2f}, NC acc={metrics['nearest_centroid_accuracy']:.2f}"
    )

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("condition-subject observation similarity")

    plt.tight_layout()
    save_current_fig(f"similarity_matrix_{key[0]}_K{key[1]}_{key[2]}_top{key[3]}")
    plt.show()
    plt.close()

def plot_top_similarity_examples(summary_df, metric="nmi", n=10):
    strong = summary_df.sort_values(f"{metric}_observed_minus_random", ascending=False).head(n)
    display(strong[[
        "analysis_type", "K", "component_ratio", "condition", "top_m",
        "decode_mean", f"{metric}_observed", f"{metric}_random_mean",
        f"{metric}_observed_minus_random", f"{metric}_z"
    ]])
    for _, r in strong.iterrows():
        key = (r["analysis_type"], int(r["K"]), r["condition"], int(r["top_m"]))
        plot_similarity_example(key)

plot_top_similarity_examples(clustering_validation_summary_df, metric="nmi", n=10)


In [ ]:

# ============================================================
# STRONGEST CASES
# ============================================================

def show_strong_cases(summary_df, metric="nmi", z_thresh=2.0, condition=None, analysis_type=None):
    df = summary_df.copy()
    if condition is not None:
        df = df[df["condition"] == condition].copy()
    if analysis_type is not None:
        df = df[df["analysis_type"] == analysis_type].copy()

    zcol = f"{metric}_z"
    diffcol = f"{metric}_observed_minus_random"

    df = df[np.isfinite(df[zcol])].copy()
    strong = df[df[zcol] >= z_thresh].sort_values(diffcol, ascending=False)

    keep = [
        "analysis_type", "K", "component_ratio", "condition", "top_m",
        "decode_mean", f"{metric}_observed", f"{metric}_random_mean",
        diffcol, zcol, f"{metric}_p_high", "feature_mode"
    ]
    return strong[[c for c in keep if c in strong.columns]]

for metric in ["nmi", "purity", "ari", "nearest_centroid_accuracy", "silhouette_true_labels"]:
    print("\n\n====", metric, "====")
    display(show_strong_cases(clustering_validation_summary_df, metric=metric, z_thresh=2.0).head(30))


In [ ]:

# ============================================================
# SAVE OUTPUTS
# ============================================================

out_long = FIG_DIR / f"spatial_enabled_clustering_validation_long_{FIT_SCOPE}.csv"
out_summary = FIG_DIR / f"spatial_enabled_clustering_validation_summary_{FIT_SCOPE}.csv"
out_skip = FIG_DIR / f"spatial_enabled_clustering_validation_skipped_{FIT_SCOPE}.csv"

clustering_validation_long_df.to_csv(out_long, index=False)
clustering_validation_summary_df.to_csv(out_summary, index=False)
clustering_validation_skipped_df.to_csv(out_skip, index=False)

print("Saved:", out_long)
print("Saved:", out_summary)
print("Saved:", out_skip)



## Interpretation guide

Use `clustering_validation_summary_df`.

A positive `*_observed_minus_random` value means the top-decoding archetype subset clusters condition-subject observations better than random archetype subsets from the same model and \(K\).

This version includes spatial AA because spatial features are built from `sXC` timecourses for each condition-subject observation rather than from node-wise loadings.
